# RefSeq lazy cache and SeqRepo bridge — rendered demonstration

This notebook is the results-first companion to [`DEMO.md`](DEMO.md). It shows the commands and captured outputs from the live native-ARM validation, so it can be read from top to bottom on GitHub without running anything.

The repository bridges versioned RefSeq range requests to an existing SeqRepo, a pre-seeded NCBI ASN Cache, a persistent GenBank BDB reader cache, or explicitly permitted remote GenBank access. Complete records can then be staged and guardedly promoted into a writable SeqRepo working instance.

> Captured outputs below are from the validated Apple-silicon environment described in the README. Generated caches and result data are intentionally not embedded in Git.

## 1. Native execution environment

The project builds and runs the NCBI C++ Toolkit inside Ubuntu 22.04 on native `linux/arm64`, using GCC 12 and C++20. The NCBI source/build tree remains inside Docker layers or BuildKit caches.

In [1]:
import subprocess

commands = [
    ["docker", "context", "show"],
    ["docker", "info", "--format", "os={{.OSType}} arch={{.Architecture}} cpus={{.NCPU}} memory={{.MemTotal}}"],
    ["docker", "run", "--rm", "--platform", "linux/arm64", "alpine:3.20", "uname", "-m"],
]
for command in commands:
    print(subprocess.check_output(command, text=True).strip())

desktop-linux
os=linux arch=aarch64 cpus=16 memory=25160732672
aarch64


## 2. Fetch an exact RefSeq range offline

The adapter accepts a **versioned** RefSeq accession and a 0-based, half-open interval. In ASN-only mode, its Docker probe runs with `--network none`, so a successful result cannot be a silent GenBank fallback.

The request below asks for `NC_000023.11[253592:253600)`.

In [2]:
command = [
    "python3", "scripts/seqrepo_bridge.py",
    "--mode", "asn",
    "fetch", "NC_000023.11", "253592", "253600",
]
print(subprocess.check_output(command, text=True).strip())

{"accession": "NC_000023.11", "aliases": ["gnl|ASM:GCF_000001305|X", "ref|NC_000023.11|", "gpp|GPC_000001315.1|", "gnl|NCBI_GENOMES|23", "gi|568815575"], "end": 253600, "sequence": "GGCTCCCA", "sequence_length": 156040895, "source": "ncbi-asn", "start": 253592}


The returned sequence is `GGCTCCCA`. The JSON also preserves the resolved record length and NCBI aliases. The complete chromosome record is cached underneath, while only the requested eight bases cross the adapter boundary.

## 3. Stage a complete record for SeqRepo

SeqRepo aliases and digests represent complete sequences, not arbitrary fragments. Promotion therefore reconstructs the complete versioned RefSeq Bioseq in bounded chunks and validates chunk order, total length, alphabet, and aliases.

This live demonstration uses the small `NM_000546.6` transcript. It stages FASTA but does **not** mutate SeqRepo because `--write` is absent.

In [3]:
command = [
    "python3", "scripts/seqrepo_bridge.py",
    "--mode", "genbank", "--allow-remote",
    "promote", "NM_000546.6",
    "--chunk-size", "500",
    "--fasta-output", "work/NM_000546.6.fasta",
]
print(subprocess.check_output(command, text=True).strip())
with open("work/NM_000546.6.fasta") as stream:
    print(stream.readline().strip())
    print(stream.readline().strip())

{"accession": "NM_000546.6", "aliases": [{"alias": "1808862652", "namespace": "gi"}, {"alias": "NM_000546.6", "namespace": "refseq"}], "committed": false, "length": 2512, "seqrepo_root": null, "source": "ncbi-genbank"}
>refseq:NM_000546.6
CTCAAAAGTCTAGAGCCACCGTCCAGGGAGCAGGTAGCTGCTGGGCTCCGGGGACACTTTGCGTTCGGGCTGGGAGCGTG


`committed: false` is the safety signal. A real write additionally requires `--seqrepo-root /path/to/seqrepo/master --write`; the bridge then rejects conflicting aliases, calls SeqRepo's writable `store()`/`commit()` API, and fetches the complete sequence back for verification. Published read-only snapshots should never be promotion targets.

## 4. Automated correctness checks

The fast suite checks coordinates, SNV/insertion/deletion anchors, deliberate off-by-one detection, result schema, versioned accessions, namespace fallback, conservative aliases, and complete-record chunk reconstruction.

In [4]:
tests = subprocess.run(
    ["python3", "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py"],
    text=True, capture_output=True, check=True,
)
print(tests.stderr.strip())
print(subprocess.check_output(["bash", "tests/smoke.sh"], text=True).strip())

..............
----------------------------------------------------------------------
Ran 14 tests in 0.000s

OK
Native ARM image smoke test passed.


## 5. Larger experiment results

The complete local experiment additionally demonstrated:

- indexed regional access to gnomAD v4.1 exomes without downloading a complete chromosome VCF;
- pre-seeded ASN Cache replay in a new network-disabled container;
- GenBank BDB cold population, new-process reuse, offline replay, and an uncached-offline negative control;
- contained-range reuse after a larger range had resolved the underlying record/blob;
- **100,000/100,000** eligible chrX REF matches offline, with zero mismatches or retrieval errors.

Run `bash scripts/run_experiment_matrix.sh` after VCF preparation and ASN hydration to regenerate normalized metrics under `results/metrics.csv`. Generated data and logs remain local by design.

## 6. Interpretation

This is demand-driven **range access**, not proof of upstream byte-range transfer. NCBI Object Manager may fetch/cache a complete record or blob even though the client receives only `[start, end)`. ASN Cache is pre-seeded and is not assumed to be write-through. GenBank BDB persistence is proven only for tested records. A fragment must never be assigned the alias or digest of its complete RefSeq record.

For every command, expected output, optional SeqRepo reads/writes, and the complete experiment matrix, continue with [`DEMO.md`](DEMO.md). See [`README.md`](README.md) for architecture and current project status.